# Setup

In [1]:
from pprint import pprint
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from transformers import set_seed
from transformers.utils import logging

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_bf16 = (device.type == 'cuda') and (torch.cuda.is_bf16_supported())

# Seed
seed = 42
set_seed(seed)

# Suppress loggings
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
logging.set_verbosity_error()
logging.disable_progress_bar()

c:\Users\yana\Desktop\Library\AI\ai-summary\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1014 14:38:52.577000 6148 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1014 14:38:52.658000 6148 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1014 14:38:53.770000 6148 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton k

# 1. Terminologies

## 1.1. Sampling

- Greedy decoding
  - Always choose the token with the highest probability.
- Temperature
  - Before softmax, divide logits by temperature $T$.
    - $p_i = softmax(\frac{z_i}{T})$
  - $T \lt 1$ → sharper distribution → more deterministic.
  - $T = 1$ → original distribution.
  - $T \gt 1$ → flatter distribution → more random/diverse.
- Top-k sampling
  - Apply temperature.
  - Keep only top-$K$ tokens.
  - Softmax.
  - Sample from the resulting distribution.
- Top-p
  - Sort tokens by prabiblity, from highest to lowest.
  - Keep the smallest token set, whose cumulative prability reaches at least $P$.
  - Sample from that set.
- Beam search
  - Predict the next token and keep the top $\mathrm{N} = \mathrm{beam\_size}$ tokens as the candidates.
  - Predict one more next token for each candidate and score.
    - Score: $P(y_{1:T}\mid x)=\prod_{t=1}^{T}P(y_t\mid x,y_{<t})$
    - Log-score: $S(y_{1:T})=\sum_{t=1}^{T}\log P(y_t\mid x,y_{<t})$
    - Normalized score: $S(y)=\frac{\sum_{t=1}^{T}\log P(y_t\mid x,y_{<t})}{T^\alpha}$
  - Keep best $\mathrm{N}$ candidates again.
  - Repeat until a generation is completed.
  > Note) Beam search is not frequently used for modern open-ended chat/story generation, because beam search tends to produce conservative/repetitive text.

## 1.2. KV-Cache

- Prefill
  - computes and stores past K/V tensors for each layer.
- Decoding
  - only the newest token is processed; past K/V values are reused.
- Memory usage
  - $2 \times L \times B \times S \times H_{KV} \times d_h \times \text{bytes/value}$
    - $2$ = K + V
    - $L$ = number of Transformer layers
    - $B$ = batch size
    - $S$ = cached sequence length
    - $H_{KV}$ = number of KV heads
    - $d_h$ = head dimension
  - Cache grows with context length and batch size.
  - GQA reduces KV-cache memory because only $H_{KV}$ key/value heads are cached, not all query heads.

# 2. API

## 2.1. Basic

> Note) `stop_strings` are avoided during `min_new_tokens`.

In [59]:
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    GenerationConfig,
)


# 1. Model
model_name = 'Qwen/Qwen2.5-0.5B-instruct'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# 2. Input
messages = [
    {'role': 'user', 'content': 'What is your name?'},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors='pt',    # tensor instead of list.
)

# 3. Generation config
generation_config = GenerationConfig(
    # Length
    max_new_tokens=16,
    min_new_tokens=4,
    stop_strings=['.'],

    # Sampling
    temperature=0.8,            # QK^T/sqrt(d) -> divide by temperature -> softmax.
    do_sample=True,             # False -> greedy decoding.
    top_k=10,                   # choose top k before softmax.
    top_p=0.9,                  # choose the smallest set whose sum >= 0.9.
    beam_size=2,                # keep 2 global candidates.

    # Repetition control
    repetition_penalty=1.1,     # if z >= 0 -> z / 1.1, z < 0 -> z x 1.1.
    no_repeat_ngram_size=3,     # forbid repeating 3 consequent tokens, e.g. "I love you" -> never appear again.

    # Settings
    use_cache=True,             # kv-cache.
    num_return_sequences=3,     # repeat and return 3 outputs.
)

# 4. Generation
model.eval()    # model to eval mode, e.g. disable dropout.

with torch.inference_mode():    # forward only -> no information stored for backprop.
    outputs = model.generate(
        **inputs,
        tokenizer=tokenizer,    # mandatory when `stop_strings` is set.
        generation_config=generation_config,
    )

answers = tokenizer.batch_decode(
    outputs,
    skip_special_tokens=True,
)

for answer in answers:
    pprint(answer)

# 5. Save and load
path = '../tmp/outputs/generation_config'
generation_config.save_pretrained(path)

generation_config = GenerationConfig.from_pretrained(path)

('system\n'
 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.\n'
 'user\n'
 'What is your name?\n'
 'assistant\n'
 'I am not a person, but an artificial intelligence language model called Qwen '
 'created')
('system\n'
 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.\n'
 'user\n'
 'What is your name?\n'
 'assistant\n'
 'I am Qwen (Qwen-001), an artificial intelligence language')
('system\n'
 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.\n'
 'user\n'
 'What is your name?\n'
 'assistant\n'
 'I am Qwen (also known as Qwen-260B),')


## 2.2. Streamer

In [55]:
from transformers import TextStreamer

streamer = TextStreamer(
    tokenizer,
    skip_prompt=True,
    skip_special_tokens=True,
)

model.eval()
with torch.inference_mode():
    model.generate(
        **inputs,
        tokenizer=tokenizer,
        streamer=streamer,
        max_length=64,
    )

I am named after the Chinese character "Qwen" which means "wisdom" or "knowledge." I was designed to be an AI language model


## 2.3. Output Scores

In [ ]:
outputs = model.generate(
    **inputs,
    tokenizer=tokenizer,
    generation_config=generation_config,
    return_dict_in_generate=True,
    output_scores=True,
)

outputs.sequences           # generated token ids.
outputs.scores              # next-token logits for each decoding step, (n_tokens, n_output_seqs, vocab_size).

transition_scores = model.compute_transition_scores(    # logits of selected tokens.
    outputs.sequences,  
    outputs.scores,
    normalize_logits=True,
)
transition_scores

tensor([[-0.4287,  0.0000, -0.5587,  0.0000, -0.6017, -0.1063,  0.0000, -1.0213,
         -2.9468, -1.6159, -1.0246,  0.0000,  0.0000, -1.8842, -1.6492, -0.1450],
        [-0.4287,  0.0000, -0.5587,  0.0000, -0.6017, -0.1063,  0.0000, -1.0213,
         -1.2280, -0.6030, -0.8604,  0.0000,  0.0000, -1.8721, -1.6772, -3.0254],
        [-0.4287,  0.0000, -0.5587,  0.0000, -0.6017, -0.1063,  0.0000, -1.0213,
         -1.0509,  0.0000,  0.0000, -2.4382, -1.8460,  0.0000, -3.1610,  0.0000]])

# 3. Batching

- batch = [A, B, C] = 3 queries -> prefill and generate together.
- Static batching
  - Each generates 10/20/30 tokens -> finish after 30 steps.
  - A new query D can't enter until 30 steps are done.
  - Waste of GPU.
- Dynamic batching
  - Static batching, but dynamically choose the batch size depending on the runtime context.
- Continuous batching
  - After A is done, D can enter right away.
  - Efficient use of GPU.

# 4. Evaluation Metrics

- Latency
  - How long one request takes.
- TTFT
  - Time to first token.
  - Measures a prefill time.
- Tokens/sec
  - generated tokens / generated time
- Throughput
  - A total amount of work the system handles per unit time.
  - e.g. 500 generated tokens/sec across all users.